In [34]:
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

# ── helper ──────────────────────────────────────────────────
def code(src): return nbf.v4.new_code_cell(src)
def md(src):   return nbf.v4.new_markdown_cell(src)

cells.append(md("# Ankh RAG — Egyptian Heritage Podcast Generator\n\nThis notebook implements a Retrieval-Augmented Generation (RAG) pipeline to create personalized podcast scripts about ancient Egyptian artifacts and sites. It uses a multimodal embedding model (DINOv2) for image-based retrieval, Qdrant as a vector database, and the Groq API for large language model (LLM) generation.\n\n## Features\n- **Multimodal RAG**: Search for information using images or text queries.\n- **Persona-Driven Generation**: Customize podcast scripts based on age group, gender, language, and knowledge level.\n- **Structured Podcast Scripting**: Generates multi-section podcast episodes with specific word count targets and narrative styles.\n- **Gradio UI**: Interactive interface for easy experimentation.\n\n## How it Works\n1. **Setup**: Installs necessary libraries and mounts Google Drive to load pre-computed embeddings.\n2. **Embeddings & Qdrant**: Loads image and text embeddings into an in-memory Qdrant vector database.\n3. **DINOv2 & Groq**: Initializes the DINOv2 model for image embedding and the Groq API client for LLM calls.\n4. **Persona Engine**: Dynamically builds a `PersonaConfig` based on user preferences.\n5. **Prompt Templates**: Constructs tailored prompts for each podcast section, incorporating retrieved context and persona details.\n6. **Podcast Generation**: Orchestrates the retrieval and generation process, calling the Groq LLM sequentially for each section.\n7. **Gradio Interface**: Provides a user-friendly web interface to interact with the RAG pipeline.\n"))

In [4]:
# ============================================================
# CELL 1 — Setup (unchanged, just add groq)
# ============================================================
cells.append(md("# Ankh RAG — Egypt Smart Tourism Assistant\n### Upgraded Podcast Script Generator (≤10 min)"))


# ============================================================
# CELL 1 — Setup & Install
# ============================================================
!pip install qdrant-client transformers accelerate gradio Pillow groq -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.2 MB/s eta 0:00:00


In [7]:
# ============================================================
# CELL 2 — Drive + Embeddings (unchanged)
# ============================================================


import pickle
from google.colab import drive
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

Connecting to Google Drive...
Mounted at /content/drive


In [9]:

# ============================================================
# CELL 2 — Mount Google Drive & Load Pre-computed Embeddings
# ============================================================




PICKLE_PATH = "/content/drive/MyDrive/EgMM-Corpus/egmm_multimodal_embeddings.pkl"

print("Loading embeddings from pickle file...")
with open(PICKLE_PATH, "rb") as f:
    qdrant_points = pickle.load(f)

print(f"Done! Total points loaded: {len(qdrant_points)}")

Loading embeddings from pickle file...
Done! Total points loaded: 4654


In [10]:
# ============================================================
# CELL 3 — Qdrant (unchanged)
# ============================================================

# ============================================================
# CELL 3 — Build Qdrant In-Memory Vector Database
# ============================================================
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(":memory:")
COLLECTION_NAME = "egmm_corpus_multimodal"

print(f"Creating collection: '{COLLECTION_NAME}'...")
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "image_vector": VectorParams(size=768, distance=Distance.COSINE),
        "text_vector":  VectorParams(size=768, distance=Distance.COSINE),
    }
)

points_to_upload = [
    PointStruct(
        id=p["id"],
        vector={
            "image_vector": p["image_vector"],
            "text_vector":  p["text_vector"]
        },
        payload=p["payload"]
    )
    for p in qdrant_points
]

print(f"Uploading {len(points_to_upload)} points...")
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points_to_upload)
print("Qdrant DB is ready!")


Creating collection: 'egmm_corpus_multimodal'...
Uploading 4654 points...
Qdrant DB is ready!


In [25]:
# ============================================================
# CELL 4 — Load DINOv2 only (Qwen removed; LLM is Groq)
# ============================================================
# ============================================================
# CELL 4 — Load AI Models
# ============================================================
# DINOv2   : image → 768-dim embedding  (unchanged, runs locally)
# LLM      : Groq API  (llama-3.3-70b-versatile)
#            ─ no GPU memory needed for LLM
#            ─ much higher quality than Qwen-0.5B for long narrative
# ============================================================

import torch
from transformers import AutoImageProcessor, AutoModel
from groq import Groq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── 4a: DINOv2 — Image Embedding Model (unchanged) ──
print("\\nLoading DINOv2 image encoder...")
DINO_MODEL_NAME = "facebook/dinov2-base"
img_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
img_model = AutoModel.from_pretrained(DINO_MODEL_NAME).to(device)
img_model.eval()
print("DINOv2 ready  (output dim: 768)")

# ── 4b: Groq API client ──
# Get your free key at https://console.groq.com
# Then set it here or via: import os; os.environ["GROQ_API_KEY"] = "gsk_..."
GROQ_API_KEY = "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"   # <─ paste your key
groq_client  = Groq(api_key=GROQ_API_KEY)

# Model options (all free on Groq):
#   "llama-3.3-70b-versatile"   ← best quality, use this
#   "llama-3.1-8b-instant"      ← fastest, lower quality
#   "mixtral-8x7b-32768"        ← large context, good alternative
GROQ_MODEL = "llama-3.3-70b-versatile"
print(f"\\nGroq LLM client ready  (model: {GROQ_MODEL})")


Using device: cpu
\nLoading DINOv2 image encoder...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 ready  (output dim: 768)
\nGroq LLM client ready  (model: llama-3.3-70b-versatile)


In [14]:
# ============================================================
# CELL 5 — Image Embedding Helper (unchanged)
# ============================================================

# ============================================================
# CELL 5 — Image → Embedding Helper  (unchanged)
# ============================================================
from PIL import Image
import torch

def get_image_embedding(image_input) -> list:
    if isinstance(image_input, str):
        image_input = Image.open(image_input)
    image_rgb = image_input.convert("RGB")
    inputs = img_processor(images=image_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = img_model(**inputs)
    cls_vector = outputs.last_hidden_state[:, 0, :]
    return cls_vector.cpu().numpy().flatten().tolist()

print("get_image_embedding() is ready!")


get_image_embedding() is ready!


In [15]:
# ============================================================
# CELL 6 — PersonaConfig dataclass
# ============================================================
cells.append(md("## Core Upgrade: Persona Engine + Podcast Script Generator"))
# ============================================================
# CELL 6 — Persona Engine
# ============================================================
# Converts (age_group, gender, language, knowledge_level)
# into a full PersonaConfig with derived narrative style.
#
# Rule-based — no model needed here.
# ============================================================

from dataclasses import dataclass
from typing import Literal

@dataclass
class PersonaConfig:
    age_group:       str   # "child" | "teen" | "adult" | "senior"
    gender:          str   # "male"  | "female" | "neutral"
    language:        str   # "en" | "ar" | "fr"
    knowledge_level: str   # "tourist" | "enthusiast" | "academic"
    # ── derived fields (set by build_persona) ──
    narrator_style:  str = ""   # "storyteller" | "professor" | "guide" | "explorer"
    vocab_level:     str = ""   # "simple" | "moderate" | "rich"
    tone:            str = ""   # "warm" | "dramatic" | "authoritative" | "playful"

# ── Lookup tables ──────────────────────────────────────────
_STYLE_MAP = {
    ("child",  "tourist"):     "storyteller",
    ("child",  "enthusiast"):  "storyteller",
    ("child",  "academic"):    "storyteller",
    ("teen",   "tourist"):     "explorer",
    ("teen",   "enthusiast"):  "explorer",
    ("teen",   "academic"):    "guide",
    ("adult",  "tourist"):     "guide",
    ("adult",  "enthusiast"):  "explorer",
    ("adult",  "academic"):    "professor",
    ("senior", "tourist"):     "guide",
    ("senior", "enthusiast"):  "professor",
    ("senior", "academic"):    "professor",
}

_VOCAB_MAP = {
    "child":  "simple",
    "teen":   "moderate",
    "adult":  "rich",
    "senior": "rich",
}

_TONE_MAP = {
    ("storyteller", "male"):    "playful",
    ("storyteller", "female"):  "warm",
    ("storyteller", "neutral"): "warm",
    ("explorer",    "male"):    "dramatic",
    ("explorer",    "female"):  "dramatic",
    ("explorer",    "neutral"): "dramatic",
    ("guide",       "male"):    "warm",
    ("guide",       "female"):  "warm",
    ("guide",       "neutral"): "warm",
    ("professor",   "male"):    "authoritative",
    ("professor",   "female"):  "authoritative",
    ("professor",   "neutral"): "authoritative",
}

def build_persona(
    age_group:       str = "adult",
    gender:          str = "male",
    language:        str = "en",
    knowledge_level: str = "tourist"
) -> PersonaConfig:
    """
    Factory function — constructs a PersonaConfig with all derived fields.

    Parameters
    ----------
    age_group       : "child" | "teen" | "adult" | "senior"
    gender          : "male"  | "female" | "neutral"
    language        : "en"    | "ar"    | "fr"
    knowledge_level : "tourist" | "enthusiast" | "academic"
    """
    style = _STYLE_MAP.get((age_group, knowledge_level), "guide")
    vocab = _VOCAB_MAP.get(age_group, "moderate")
    tone  = _TONE_MAP.get((style, gender), "warm")

    return PersonaConfig(
        age_group       = age_group,
        gender          = gender,
        language        = language,
        knowledge_level = knowledge_level,
        narrator_style  = style,
        vocab_level     = vocab,
        tone            = tone,
    )


# ── Quick test ──────────────────────────────────────────────
test_personas = [
    build_persona("child",  "male",    "en", "tourist"),
    build_persona("adult",  "female",  "en", "enthusiast"),
    build_persona("senior", "neutral", "ar", "academic"),
]
for p in test_personas:
    print(f"[{p.age_group}/{p.gender}/{p.knowledge_level}]  "
          f"→  style={p.narrator_style}  vocab={p.vocab_level}  tone={p.tone}")

print("\\nbuild_persona() is ready!")


[child/male/tourist]  →  style=storyteller  vocab=simple  tone=playful
[adult/female/enthusiast]  →  style=explorer  vocab=rich  tone=dramatic
[senior/neutral/academic]  →  style=professor  vocab=rich  tone=authoritative
\nbuild_persona() is ready!


In [16]:
# ============================================================
# CELL 7 — Prompt Templates
# ============================================================
# ============================================================
# CELL 7 — Prompt Template Builder
# ============================================================
# Each persona combination gets its OWN system prompt.
# Do NOT merge all personas into one prompt — the model
# will average them out and produce mediocre output for every
# persona instead of great output for the target audience.
#
# Word targets for ≤10 min podcast (@ 130 wpm):
#   intro_hook            :  130–180 words  (1 min)
#   historical_background :  380–450 words  (3 min)
#   construction_builders :  300–380 words  (2.5 min)
#   legends_stories       :  260–320 words  (2 min)
#   outro                 :  130–170 words  (1 min)
#   TOTAL                 : 1200–1500 words  (≤10 min)
# ============================================================

# ── Section structure (constant) ──────────────────────────
PODCAST_SECTIONS = [
    "intro_hook",
    "historical_background",
    "construction_builders",
    "legends_stories",
    "outro",
]

SECTION_WORD_TARGETS = {
    "intro_hook":            (130, 180),
    "historical_background": (380, 450),
    "construction_builders": (300, 380),
    "legends_stories":       (260, 320),
    "outro":                 (130, 170),
}

# ── Narrator persona descriptions ─────────────────────────
_NARRATOR_VOICE = {
    # (narrator_style, tone) → description for system prompt
    ("storyteller", "playful"): (
        "You are an enthusiastic storyteller narrating to children aged 7-12. "
        "Use short sentences. Replace all dates with vivid phrases like "
        "'thousands of years ago' or 'before your great-great-great-grandparents were born'. "
        "Use comparisons to everyday objects (like a school bus, a football field). "
        "Add 2-3 'wow' facts. Never use words with more than 3 syllables if a simpler word exists."
    ),
    ("storyteller", "warm"): (
        "You are a warm, nurturing storyteller narrating to girls aged 7-12. "
        "Focus on the beauty, colors, and human stories behind the artifact. "
        "Use gentle, vivid imagery. Short sentences. Replace dates with 'long, long ago'. "
        "Add small emotional moments — imagine the people who lived there."
    ),
    ("explorer", "dramatic"): (
        "You are an adventurous explorer-narrator speaking to teenagers aged 13-17. "
        "Use a documentary tone — think National Geographic. "
        "Include actual dates and names but frame them dramatically. "
        "Ask rhetorical questions. Build suspense before reveals."
    ),
    ("guide", "warm"): (
        "You are a knowledgeable, friendly Egyptian tour guide speaking to adult tourists. "
        "Balance historical facts with human interest. "
        "Use vivid scene-setting. Speak as if the listener is standing in front of the monument. "
        "Include dates, historical figures, and one surprising fact they won't find in guidebooks."
    ),
    ("professor", "authoritative"): (
        "You are a distinguished Egyptologist delivering an academic audio lecture. "
        "Use precise dates, architectural terminology, political context, and scholarly analysis. "
        "Reference the historical significance within the broader context of Egyptian history. "
        "Maintain narrative tension — this is a story, not a dry lecture. "
        "Speak to an educated adult who wants depth, not simplification."
    ),
}

def _get_narrator_voice(persona: PersonaConfig) -> str:
    key = (persona.narrator_style, persona.tone)
    # fallback to guide/warm if exact combo not found
    return _NARRATOR_VOICE.get(key, _NARRATOR_VOICE[("guide", "warm")])


# ── Section-specific instructions ─────────────────────────
_SECTION_INSTRUCTIONS = {
    "intro_hook": (
        "Open with a dramatic hook — a single vivid scene, sensory detail, or "
        "rhetorical question that places the listener at this site. "
        "Name the artifact/site explicitly. End the intro with a promise: "
        "'Today you will discover...' type transition."
    ),
    "historical_background": (
        "Tell the origin story. When was it built or discovered? "
        "What was the historical period? What empire, dynasty, or civilization? "
        "What was happening in Egypt and the world at that time? "
        "Make it feel like a story unfolding, not a list of facts."
    ),
    "construction_builders": (
        "Focus on WHO built it and HOW. Name the pharaoh, architect, or civilization. "
        "Describe the construction — scale, materials, engineering challenges. "
        "Include one concrete measurement or scale comparison if available in context. "
        "End with WHY they built it — purpose, religion, or power."
    ),
    "legends_stories": (
        "Share the myths, legends, and human stories connected to this site. "
        "What did ancient Egyptians believe about it? What mysteries still exist? "
        "Include any famous historical events that happened here. "
        "This section should feel like storytelling, not a history textbook."
    ),
    "outro": (
        "Bring the listener back to the present. "
        "What does this site mean today? What can visitors experience? "
        "End with a memorable closing thought, poetic reflection, or call to visit. "
        "Do NOT introduce new historical facts in the outro."
    ),
}


def build_section_prompt(
    section_name: str,
    persona: PersonaConfig,
    context_chunks: str,
    entity_name: str = "this ancient Egyptian site",
) -> str:
    """
    Builds the complete prompt for a single podcast section.

    Parameters
    ----------
    section_name  : one of PODCAST_SECTIONS
    persona       : PersonaConfig from build_persona()
    context_chunks: RAG-retrieved text, already joined
    entity_name   : resolved name of the artifact/site
    """
    word_min, word_max = SECTION_WORD_TARGETS[section_name]
    narrator_voice    = _get_narrator_voice(persona)
    section_task      = _SECTION_INSTRUCTIONS[section_name]

    prompt = f"""You are writing one section of a podcast script about {entity_name}.

== NARRATOR VOICE ==
{narrator_voice}

== YOUR TASK FOR THIS SECTION: {section_name.upper().replace("_", " ")} ==
{section_task}

== VERIFIED FACTUAL CONTEXT (from knowledge base) ==
Use ONLY these facts. Do NOT invent dates, names, measurements, or events
that are not present in the context below. If a fact is not here, do not include it.
If something is unknown, say "historians believe..." or "legend tells us...".

{context_chunks}

== OUTPUT FORMAT ==
- Write ONLY flowing prose. No bullet points. No headers. No section titles.
- Word count: {word_min}–{word_max} words (strictly enforced).
- This section will be read aloud — write for the ear, not the eye.
- Use natural spoken-word rhythm. Short sentences for drama. Longer ones for context.
- Language: {persona.language.upper()}
- Vocabulary level: {persona.vocab_level}

Write the {section_name.replace("_", " ")} section now:"""

    return prompt


print("Prompt template system ready!")
print(f"Sections: {PODCAST_SECTIONS}")
print(f"Total target: {sum(lo for lo,_ in SECTION_WORD_TARGETS.values())}–"
      f"{sum(hi for _,hi in SECTION_WORD_TARGETS.values())} words  (~≤10 min)")


Prompt template system ready!
Sections: ['intro_hook', 'historical_background', 'construction_builders', 'legends_stories', 'outro']
Total target: 1200–1500 words  (~≤10 min)


In [17]:
# ============================================================
# CELL 8 — RAG retriever
# ============================================================
# ============================================================
# CELL 8 — RAG Retriever
# ============================================================
# Retrieves top-k chunks from Qdrant and resolves entity name.
# Returns clean context string + entity name for the prompts.
# ============================================================

def retrieve_context(
    query_vector: list,
    search_type:  str = "image_vector",
    limit:        int = 5,
) -> tuple[str, str]:
    """
    Search Qdrant, deduplicate, and return:
        context_str : joined chunk text, each prefixed with 'Fact: '
        entity_name : best-guess name from top result payload
    """
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using=search_type,
        limit=limit,
    ).points

    if not results:
        return "No context found in knowledge base.", "this Egyptian artifact"

    # ── Extract entity name from top result ──────────────
    top_payload  = results[0].payload
    entity_name  = (
        top_payload.get("Concept_Name")
        or top_payload.get("concept_name")
        or top_payload.get("name")
        or "this Egyptian artifact"
    )

    # ── Collect + deduplicate chunks ─────────────────────
    raw_chunks = []
    seen = set()
    for r in results:
        chunk = r.payload.get("Text_Chunk") or r.payload.get("Full_Description") or ""
        chunk = chunk.strip()
        if chunk and chunk not in seen:
            seen.add(chunk)
            raw_chunks.append(chunk)

    if not raw_chunks:
        return "No usable text found in retrieved results.", entity_name

    context_str = "\\n\\n".join(f"Fact: {c}" for c in raw_chunks)
    return context_str, entity_name


print("retrieve_context() is ready!")


retrieve_context() is ready!


In [26]:
# ============================================================
# CELL 9 — Groq LLM caller
# ============================================================
# ============================================================
# CELL 9 — Groq LLM Caller
# ============================================================
# Sends a prompt to Groq and returns the generated text.
# Includes:
#   - temperature tuned per section type
#   - word-count post-validation (warns but does not crash)
#   - hallucination guard reminder in every system message
# ============================================================

def _section_temperature(section_name: str) -> float:
    """
    Creative sections get higher temperature.
    Factual sections get lower temperature.
    """
    return {
        "intro_hook":            0.75,
        "historical_background": 0.35,
        "construction_builders": 0.35,
        "legends_stories":       0.70,
        "outro":                 0.65,
    }.get(section_name, 0.5)


HALLUCINATION_GUARD_SYSTEM = (
    "CRITICAL RULE: You must ONLY use facts that appear in the provided context. "
    "If a date, name, or measurement is not in the context, do NOT include it. "
    "If something is uncertain, use hedging language: 'historians believe...', "
    "'according to ancient records...', 'legend suggests...'. "
    "Never invent specific numbers, proper names, or historical events."
)


def call_groq(
    prompt:       str,
    section_name: str,
    max_tokens:   int = 700,
) -> str:
    """
    Call Groq API and return the generated text.

    Parameters
    ----------
    prompt       : full section prompt from build_section_prompt()
    section_name : used to set temperature
    max_tokens   : hard cap; 700 ≈ 500 words (safe margin above section targets)
    """
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=_section_temperature(section_name),
            max_tokens=max_tokens,
        )
        generated = response.choices[0].message.content.strip()

        # ── Post-generation word count check ──
        word_count = len(generated.split())
        w_min, w_max = SECTION_WORD_TARGETS[section_name]
        if word_count < w_min:
            print(f"  ⚠ [{section_name}] Generated {word_count} words — below target ({w_min})")
        elif word_count > w_max + 50:   # allow 50-word overflow
            print(f"  ⚠ [{section_name}] Generated {word_count} words — above target ({w_max})")
        else:
            print(f"  ✓ [{section_name}] {word_count} words  (target: {w_min}–{w_max})")

        return generated

    except Exception as e:
        print(f"  ✗ [{section_name}] Groq API error: {e}")
        return f"[Generation failed for {section_name}: {e}]"


print("call_groq() is ready!")


call_groq() is ready!


In [27]:
# ============================================================
# CELL 10 — Full podcast generator
# ============================================================
# ============================================================
# CELL 10 — Full Podcast Script Generator
# ============================================================
# Orchestrates the complete pipeline:
#
#   image / text query
#       ↓
#   DINOv2 embedding  (image path)  OR  text embedding (text path)
#       ↓
#   Qdrant retrieval  →  context_str + entity_name
#       ↓
#   PersonaConfig  →  5 section prompts
#       ↓
#   Groq LLM  (one API call per section, sequential)
#       ↓
#   PodcastScript  (validated, assembled)
# ============================================================
import time
from dataclasses import dataclass, field

@dataclass
class PodcastScript:
    entity_name:        str
    persona:            PersonaConfig
    sections:           dict = field(default_factory=dict)  # section_name → text
    total_word_count:   int  = 0
    estimated_duration: float = 0.0   # minutes

    def assemble(self) -> str:
        """Return the full script as a single string with section markers."""
        parts = []
        for section in PODCAST_SECTIONS:
            text = self.sections.get(section, "")
            if text:
                parts.append(f"--- {section.upper().replace('_', ' ')} ---\n{text}")
        return "\n\n".join(parts)


def generate_podcast_script(
    image_input      = None,       # PIL.Image or str path (for image query)
    text_query       : str = None, # text query if no image
    age_group        : str = "adult",
    gender           : str = "male",
    language         : str = "en",
    knowledge_level  : str = "tourist",
    rag_limit        : int = 5,
    verbose          : bool = True,
) -> PodcastScript:
    """
    Main entry point. Provide EITHER image_input OR text_query.

    Parameters
    ----------
    image_input    : PIL.Image or str file path
    text_query     : string query (used if no image provided)
    age_group      : "child" | "teen" | "adult" | "senior"
    gender         : "male"  | "female" | "neutral"
    language       : "en"    | "ar"    | "fr"
    knowledge_level: "tourist" | "enthusiast" | "academic"
    rag_limit      : number of Qdrant results to retrieve (5 recommended)
    verbose        : print progress
    """
    print("=" * 60)
    print("ANKH PODCAST GENERATOR — starting pipeline")
    print("=" * 60)

    # ── Step 1: Build PersonaConfig ───────────────────────
    persona = build_persona(age_group, gender, language, knowledge_level)
    if verbose:
        print(f"\n[Persona] style={persona.narrator_style}  "
              f"vocab={persona.vocab_level}  tone={persona.tone}")

    # ── Step 2: Generate query vector ─────────────────────
    if image_input is not None:
        if verbose: print("\n[Step 2] Embedding image via DINOv2...")
        query_vector = get_image_embedding(image_input)
        search_type  = "image_vector"
    elif text_query:
        # For text queries we reuse the stored text_vectors via a simple
        # keyword match on payload — a proper text embedding model would
        # be better, but this works for the current dataset.
        # Quick workaround: find the point whose Concept_Name best matches.
        if verbose: print(f"\n[Step 2] Text query mode: '{text_query}'")
        # Use the first matching point\'s text_vector as the query
        matched = [
            p for p in qdrant_points
            if text_query.lower() in str(p["payload"].get("Concept_Name","")).lower()
            or text_query.lower() in str(p["payload"].get("Text_Chunk","")).lower()
        ]
        if matched:
            query_vector = matched[0]["text_vector"]
            if verbose: print(f"   Found match: {matched[0]['payload'].get('Concept_Name','?')}")
        else:
            # Fall back to first point in DB (graceful degradation)
            query_vector = qdrant_points[0]["text_vector"]
            if verbose: print("   No exact match — using first DB entry as fallback")
        search_type = "text_vector"
    else:
        raise ValueError("Provide either image_input or text_query")

    # ── Step 3: Retrieve context from Qdrant ──────────────
    if verbose: print("\n[Step 3] Retrieving context from Qdrant...")
    context_str, entity_name = retrieve_context(query_vector, search_type, rag_limit)
    if verbose:
        print(f"   Entity : {entity_name}")
        print(f"   Context: {len(context_str.split())} words retrieved")

    # ── Step 4: Generate each section via Groq ────────────
    if verbose: print("\n[Step 4] Generating podcast sections via Groq LLM...")
    script = PodcastScript(entity_name=entity_name, persona=persona)

    for section in PODCAST_SECTIONS:
        if verbose: print(f"\n  Generating: {section}...")
        t0 = time.time()

        prompt = build_section_prompt(
            section_name  = section,
            persona       = persona,
            context_chunks= context_str,
            entity_name   = entity_name,
        )
        text = call_groq(prompt, section_name=section)
        script.sections[section] = text

        elapsed = time.time() - t0
        if verbose: print(f"     ({elapsed:.1f}s)")

        # Small delay to avoid Groq rate limit (free tier: 30 req/min)
        time.sleep(1)

    # ── Step 5: Compute stats ─────────────────────────────
    full_text = " ".join(script.sections.values())
    script.total_word_count   = len(full_text.split())
    script.estimated_duration = script.total_word_count / 130   # minutes @ 130 wpm

    print("\n" + "=" * 60)
    print(f"DONE  |  {script.total_word_count} words  "
          f"|  ~{script.estimated_duration:.1f} min estimated")
    print("=" * 60)

    return script


print("generate_podcast_script() is ready!")


generate_podcast_script() is ready!


In [28]:
# ============================================================
# CELL 11 — Legacy wrapper (backward compatibility)
# ============================================================
# ============================================================
# CELL 11 — Legacy RAG Functions (Backward Compatibility)
# ============================================================
# These keep the original get_strict_summary() and
# get_personalized_story() working so old Gradio cells don\'t break.
#
# BUT: the recommended way is now generate_podcast_script().
# ============================================================

from IPython.display import display, Markdown

def get_strict_summary(query_vector: list, search_type: str = "image_vector", limit: int = 3):
    """Legacy: academic description (short, not a podcast)."""
    context_str, entity_name = retrieve_context(query_vector, search_type, limit)

    prompt = (
        f"You are an expert Egyptologist Tour Guide. "
        f"Provide a comprehensive, structured explanation of {entity_name} "
        f"using ONLY the facts below.\\n\\n"
        f"{context_str}\\n\\n"
        f"Write clear, readable paragraphs. English only. ~200 words."
    )
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0.3,
            max_tokens=400,
        )
        return response.choices[0].message.content.strip(), context_str
    except Exception as e:
        return f"Generation failed: {e}", context_str


def get_personalized_story(
    query_vector: list,
    search_type:  str = "image_vector",
    user_persona: str = "man_adult",
    limit:        int = 3
):
    """
    Legacy: short personalized story (2 paragraphs).
    Keeps old persona keys: boy_child | girl_child | man_adult | woman_adult
    """
    # Map old keys to new PersonaConfig
    _LEGACY_MAP = {
        "boy_child":   ("child",  "male"),
        "girl_child":  ("child",  "female"),
        "man_adult":   ("adult",  "male"),
        "woman_adult": ("adult",  "female"),
    }
    age, gender = _LEGACY_MAP.get(user_persona, ("adult", "male"))
    persona     = build_persona(age, gender, "en", "tourist")

    context_str, entity_name = retrieve_context(query_vector, search_type, limit)

    narrator = _get_narrator_voice(persona)
    prompt = (
        f"{narrator}\\n\\n"
        f"Write exactly 2 short paragraphs (2-3 sentences each) about {entity_name}. "
        f"Use ONLY these facts:\\n{context_str}\\n\\n"
        f"No bullet points. No headers. Flowing prose. English only."
    )
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": HALLUCINATION_GUARD_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0.6,
            max_tokens=350,
        )
        return response.choices[0].message.content.strip(), context_str
    except Exception as e:
        return f"Story generation failed: {e}", context_str


print("Legacy functions (get_strict_summary + get_personalized_story) ready!")
print("Recommended: use generate_podcast_script() for full podcast output.")


Legacy functions (get_strict_summary + get_personalized_story) ready!
Recommended: use generate_podcast_script() for full podcast output.


In [29]:
# ============================================================
# CELL 12 — Pipeline wrapper
# ============================================================
# ============================================================
# CELL 12 — ankh_rag_pipeline  (updated to support podcast mode)
# ============================================================

def ankh_rag_pipeline(
    image_input,
    mode:    str = "description",
    persona: str = "man_adult",
) -> tuple:
    """
    Original interface — unchanged for Gradio compatibility.

    mode:
        "description" → academic summary (fast, ~200 words)
        "story"       → short personalized story (2 paragraphs)
        "podcast"     → full ~10-min podcast script (PodcastScript object
                         returned as text via assemble())

    persona (for description/story): "man_adult" | "woman_adult" | "boy_child" | "girl_child"
    persona (for podcast):           same keys, mapped internally
    """
    print("=" * 55)
    print(f"Mode: {mode.upper()}  |  Persona: {persona}")
    print("=" * 55)

    query_vector = get_image_embedding(image_input)
    print(f"Embedding dim: {len(query_vector)}")

    if mode == "podcast":
        _LEGACY_MAP = {
            "boy_child":   ("child",  "male",   "en", "tourist"),
            "girl_child":  ("child",  "female", "en", "tourist"),
            "man_adult":   ("adult",  "male",   "en", "tourist"),
            "woman_adult": ("adult",  "female", "en", "tourist"),
        }
        age, gender, lang, kl = _LEGACY_MAP.get(persona, ("adult","male","en","tourist"))
        script = generate_podcast_script(
            image_input     = image_input,
            age_group       = age,
            gender          = gender,
            language        = lang,
            knowledge_level = kl,
        )
        return script.assemble(), f"Entity: {script.entity_name}\\n~{script.estimated_duration:.1f} min | {script.total_word_count} words"

    elif mode == "story":
        return get_personalized_story(query_vector, "image_vector", persona)
    else:
        return get_strict_summary(query_vector, "image_vector")


print("ankh_rag_pipeline() updated!")


ankh_rag_pipeline() updated!


In [33]:
# ============================================================
# CELL 13 — Gradio UI (updated)
# ============================================================
# ============================================================
# CELL 13 — Gradio Interactive Interface (Updated)
# ============================================================

import gradio as gr
from PIL import Image

def gradio_handler(image, mode, persona, age_group, gender, language, knowledge_level):
    if image is None:
        return "Please upload an image first.", "No image provided.", "—"
    try:
        if mode == "podcast":
            # Use full persona controls for podcast mode
            script = generate_podcast_script(
                image_input     = image,
                age_group       = age_group,
                gender          = gender,
                language        = language,
                knowledge_level = knowledge_level,
            )
            assembled = script.assemble()
            context_info = f"Entity: {script.entity_name} | {script.total_word_count} words | ~{script.estimated_duration:.1f} min"
            return assembled, context_info, f"Persona: {script.persona.narrator_style} / {script.persona.tone} / vocab={script.persona.vocab_level}"
        else:
            response, context = ankh_rag_pipeline(image, mode, persona)
            return response, context, f"DINOv2 768-dim | {len(qdrant_points)} DB points"
    except Exception as e:
        err = f"Error: {str(e)}"
        return err, err, "—"


with gr.Blocks(title="Ankh RAG — Egyptian Heritage Guide", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏛️ Ankh RAG — Egyptian Heritage Podcast Generator
    Upload an image of an Egyptian artifact or historic site.
    Choose **Podcast** mode for a full narrated ~10-minute script.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Upload Artifact / Site Image")

            mode_radio = gr.Radio(
                choices=["description", "story", "podcast"],
                value="description",
                label="Response Mode",
                info="'description'=academic | 'story'=short narrative | 'podcast'=full ~10min script"
            )

            gr.Markdown("### Quick Persona (for description/story modes)")
            persona_dropdown = gr.Dropdown(
                choices=["man_adult", "woman_adult", "boy_child", "girl_child"],
                value="man_adult",
                label="Quick Persona"
            )

            gr.Markdown("### Full Persona Controls (for podcast mode)")
            age_dd  = gr.Dropdown(["child","teen","adult","senior"], value="adult", label="Age Group")
            gen_dd  = gr.Dropdown(["male","female","neutral"], value="male", label="Gender")
            lang_dd = gr.Dropdown(["en","ar","fr"], value="en", label="Language")
            kl_dd   = gr.Dropdown(["tourist","enthusiast","academic"], value="tourist", label="Knowledge Level")

            submit_btn = gr.Button("Generate 🔍", variant="primary")

        with gr.Column(scale=1):
            output_response = gr.Textbox(label="Generated Output", lines=25)
            output_context  = gr.Textbox(label="Context / Stats", lines=4)
            output_info     = gr.Textbox(label="Persona Info", lines=2, interactive=False)

    submit_btn.click(
        fn=gradio_handler,
        inputs=[img_input, mode_radio, persona_dropdown, age_dd, gen_dd, lang_dd, kl_dd],
        outputs=[output_response, output_context, output_info]
    )

    gr.Markdown("""
    ---
    **Pipeline:** Image → DINOv2 embedding → Qdrant retrieval → PersonaConfig → 5-section Groq LLM → Podcast script
    """)

print("Launching Gradio...")
demo.launch(debug=True, share=True)


/tmp/ipykernel_4603/1744282837.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Ankh RAG — Egyptian Heritage Guide", theme=gr.themes.Soft()) as demo:


Launching Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://42bca2ac8f4c2aa9b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Mode: DESCRIPTION  |  Persona: man_adult
Embedding dim: 768


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Mode: DESCRIPTION  |  Persona: man_adult
Embedding dim: 768
Mode: STORY  |  Persona: man_adult
Embedding dim: 768
ANKH PODCAST GENERATOR — starting pipeline

[Persona] style=guide  vocab=rich  tone=warm

[Step 2] Embedding image via DINOv2...

[Step 3] Retrieving context from Qdrant...
   Entity : Monastery of Saint Anthony
   Context: 767 words retrieved

[Step 4] Generating podcast sections via Groq LLM...

  Generating: intro_hook...
  ⚠ [intro_hook] Generated 113 words — below target (130)
     (0.9s)

  Generating: historical_background...
  ✓ [historical_background] 391 words  (target: 380–450)
     (1.8s)

  Generating: construction_builders...
  ⚠ [construction_builders] Generated 285 words — below target (300)
     (1.3s)

  Generating: legends_stories...
  ✓ [legends_stories] 283 words  (target: 260–320)
     (1.4s)

  Generating: outro...
  ⚠ [outro] Generated 101 words — below target (130)
     (1.0s)

DONE  |  1173 words  |  ~9.0 min estimated
ANKH PODCAST GENERATOR — star

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

ANKH PODCAST GENERATOR — starting pipeline

[Persona] style=storyteller  vocab=simple  tone=warm

[Step 2] Embedding image via DINOv2...

[Step 3] Retrieving context from Qdrant...
   Entity : Giza pyramid complex
   Context: 895 words retrieved

[Step 4] Generating podcast sections via Groq LLM...

  Generating: intro_hook...
  ⚠ [intro_hook] Generated 86 words — below target (130)
     (0.7s)

  Generating: historical_background...
  ✓ [historical_background] 498 words  (target: 380–450)
     (2.2s)

  Generating: construction_builders...
  ✓ [construction_builders] 309 words  (target: 300–380)
     (1.8s)

  Generating: legends_stories...
  ✓ [legends_stories] 303 words  (target: 260–320)
     (1.5s)

  Generating: outro...
  ⚠ [outro] Generated 109 words — below target (130)
     (0.8s)

DONE  |  1305 words  |  ~10.0 min estimated
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://42bca2ac8f4c2aa9b4.gradio.live


In [31]:
# ============================================================
# CELL 14 — Direct test (no Gradio)
# ============================================================

# ============================================================
# CELL 14 — Direct Test: generate a podcast from a stored vector
# ============================================================
# This tests the full pipeline without needing a real image upload.
# Uses the first stored embedding from the DB as the query vector.
# ============================================================

from PIL import Image as PILImage
import numpy as np

# Simulate an image query using the first stored image_vector
# (In real use, this would be a user-uploaded image)
test_vector = qdrant_points[0]["image_vector"]

# Manually call retrieve_context to see what entity we\'re working with
test_context, test_entity = retrieve_context(test_vector, "image_vector", limit=5)
print(f"Testing with entity: {test_entity}")
print(f"Context preview: {test_context[:300]}...")
print()

# Build a persona
test_persona = build_persona("adult", "female", "en", "enthusiast")
print(f"Persona: {test_persona}")
print()

# Generate one section to verify everything works before running all 5
print("--- Testing single section (intro_hook) ---")
test_prompt = build_section_prompt(
    section_name  = "intro_hook",
    persona       = test_persona,
    context_chunks= test_context,
    entity_name   = test_entity,
)
test_output = call_groq(test_prompt, section_name="intro_hook")
print()
print(test_output)


Testing with entity: Bab al-Nasr (Cairo)
Context preview: Fact: # Wikipedia: Bab al-Nasr (Cairo)
Bab al-Nasr (Arabic: باب النصر, lit. 'Gate of Victory'), is one of three remaining gates in the historic city wall of Cairo, the capital of Egypt. The gate's construction is dated to 1087 and was ordered by Badr al-Jamali, a Fatimid vizier. It is located at the...

Persona: PersonaConfig(age_group='adult', gender='female', language='en', knowledge_level='enthusiast', narrator_style='explorer', vocab_level='rich', tone='dramatic')

--- Testing single section (intro_hook) ---
  ⚠ [intro_hook] Generated 87 words — below target (130)

Imagine standing before a centuries-old gateway, its stone walls bearing the weight of history. What secrets lie behind the imposing Bab al-Nasr, the Gate of Victory, in the ancient city of Cairo? As you approach this fortified entrance, the whispers of the past begin to unfold. Constructed in 1087, this gate has witnessed the rise and fall of empires. Today, you 

In [ ]:
cells.append(code('''\
# ============================================================
# CELL 15 — Full Podcast Generation Test
# ============================================================
# Run this only after Cell 14 succeeds.
# Generates all 5 sections (~5 Groq API calls, ~15-20 seconds).
# ============================================================

# Option A: generate from a stored vector (no image needed for testing)
# We create a tiny 1x1 white image and manually override with stored vector
# by calling generate_podcast_script with a text query instead

full_script = generate_podcast_script(
    text_query      = qdrant_points[0]["payload"].get("Concept_Name", "egypt"),
    age_group       = "adult",
    gender          = "female",
    language        = "en",
    knowledge_level = "enthusiast",
    verbose         = True,
)

print("\\n\\n" + "=" * 60)
print("FULL ASSEMBLED SCRIPT")
print("=" * 60)
print(full_script.assemble())
'''))
